# EXP-02 Indiana/OpenI external evaluation
Research-only, paper-aligned subset evaluation. Run inference and clinical metrics only after explicitly enabling the gates below.

## Scope and safety
This notebook does not train or modify CXR-LLaVA. It evaluates report generation only. Generated text is experimental model output, not a verified clinical finding or medical advice. The full dataset is opt-in and disabled by default.

In [ ]:
# Configuration gates: change explicitly before expensive work.
RUN_INFERENCE = False
RUN_SUBSET_SIZE = 10
RUN_FULL_DATASET = False
SEED = 42
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 42
MODEL_ID = 'ECOFRI/CXR-LLAVA-v2'
GITHUB_REPO_URL = 'https://github.com/hanhpm/CXR_LLaVA_Improvement.git'
GITHUB_BRANCH = 'master'
print('RUN_INFERENCE:', RUN_INFERENCE, '| RUN_SUBSET_SIZE:', RUN_SUBSET_SIZE)

In [ ]:
from pathlib import Path
import os, subprocess, sys, platform, time, csv, json, shutil, hashlib
import pandas as pd
from PIL import Image
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DATASETS_ROOT = Path('/content/drive/MyDrive/ResearchLab/Notebook/datasets')
assert DATASETS_ROOT.exists(), DATASETS_ROOT
PROJECT_DIR = Path('/content/CXR_LLaVA_Improvement')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--branch',GITHUB_BRANCH,GITHUB_REPO_URL,str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR, '| datasets:', DATASETS_ROOT)

In [ ]:
# Inspect first; change only these two variables if uploaded folder names differ.
print('Dataset folders:')
for p in sorted(DATASETS_ROOT.iterdir()): print(' -', p.name)
NLMCXR_IMAGE_DIR = DATASETS_ROOT / 'NLMCXR_png'
NLMCXR_REPORT_DIR = DATASETS_ROOT / 'NLMCXR_reports'
assert NLMCXR_IMAGE_DIR.exists(), NLMCXR_IMAGE_DIR
assert NLMCXR_REPORT_DIR.exists(), NLMCXR_REPORT_DIR

In [ ]:
EXP_ROOT = DATASETS_ROOT / 'CXR_LLaVA_EXP02_Indiana_External_Eval'
MANIFEST_DIR=EXP_ROOT/'manifests'; SUBSET_DIR=EXP_ROOT/'subsets'; PREDICTION_DIR=EXP_ROOT/'predictions'
CHEXPERT_INPUT_DIR=EXP_ROOT/'chexpert_inputs'; CHEXPERT_LABEL_DIR=EXP_ROOT/'chexpert_labels'; METRIC_DIR=EXP_ROOT/'metrics'
ERROR_DIR=EXP_ROOT/'error_analysis'; LOG_DIR=EXP_ROOT/'logs'; DRIVE_REPORT_DIR=EXP_ROOT/'reports'
for p in [MANIFEST_DIR,SUBSET_DIR,PREDICTION_DIR,CHEXPERT_INPUT_DIR,CHEXPERT_LABEL_DIR,METRIC_DIR,ERROR_DIR,LOG_DIR,DRIVE_REPORT_DIR]: p.mkdir(parents=True,exist_ok=True)
print('EXP_ROOT:', EXP_ROOT)

In [ ]:
# Environment and raw inventory (lightweight; no model inference).
import torch, transformers, bitsandbytes as bnb
png_files=sorted(NLMCXR_IMAGE_DIR.rglob('*.png')); xml_files=sorted(NLMCXR_REPORT_DIR.rglob('*.xml'))
env = {'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()), 'python':platform.python_version(), 'pytorch':torch.__version__, 'transformers':transformers.__version__, 'bitsandbytes':getattr(bnb,'__version__','unknown'), 'cuda':torch.version.cuda, 'cuda_available':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', 'model':MODEL_ID, 'quantization':'4-bit NF4', 'compute_dtype':'float16', 'seed':SEED}
env.update({'png_count':len(png_files),'xml_count':len(xml_files)})
(LOG_DIR/'experiment_environment.txt').write_text(json.dumps(env,indent=2),encoding='utf-8')
(LOG_DIR/'nlmcxr_inventory.txt').write_text(f'PNG: {len(png_files)}\nXML: {len(xml_files)}\n',encoding='utf-8')
assert png_files and xml_files
print(env)

In [ ]:
from xml.etree import ElementTree as ET
def clean_text(v): return ' '.join(str(v or '').split())
def parse_report(path):
    root=ET.parse(path).getroot(); sections={}
    for n in root.findall('.//AbstractText'):
        label=clean_text(n.attrib.get('Label')).upper(); text=clean_text(' '.join(n.itertext()))
        if label: sections[label]=text
    images=[]
    for n in root.findall('.//parentImage'):
        image_id=clean_text(n.attrib.get('id')); cap=n.find('caption')
        caption=clean_text(' '.join(cap.itertext())) if cap is not None else ''
        if image_id: images.append((image_id,caption))
    f=sections.get('FINDINGS',''); i=sections.get('IMPRESSION',''); gt=' '.join(x for x in [f,i] if x)
    source='findings+impression' if f and i else 'findings_only' if f else 'impression_only' if i else 'missing'
    return sections,images,gt,source
def classify_view(caption):
    c=caption.lower()
    if any(x in c for x in ['lateral']): return 'lateral','caption:lateral'
    if any(x in c for x in ['pa','ap','frontal','posteroanterior','anteroposterior']): return 'frontal','caption:'+c
    return 'unknown',caption
image_index={p.stem:p for p in png_files}; rows=[]; issues=[]
for xp in xml_files:
    try: sections,images,gt,source=parse_report(xp)
    except Exception as e: issues.append({'xml_path':str(xp),'issue':repr(e)}); continue
    for image_id,caption in images:
        ip=image_index.get(image_id); view,evidence=classify_view(caption)
        if ip is None: issues.append({'xml_path':str(xp),'image_id':image_id,'issue':'missing_png'})
        if not gt: issues.append({'xml_path':str(xp),'image_id':image_id,'issue':'missing_report_text'})
        rows.append({'report_id':xp.stem,'xml_path':str(xp),'image_id':image_id,'image_path':str(ip) if ip else '', 'image_exists':bool(ip and ip.is_file()),'caption':caption,'view_guess':view,'view_evidence':evidence,'gt_findings':sections.get('FINDINGS',''),'gt_impression':sections.get('IMPRESSION',''),'ground_truth_report':gt,'report_text_source':source})
manifest=pd.DataFrame(rows); issues_df=pd.DataFrame(issues)
manifest.to_csv(MANIFEST_DIR/'indiana_full_manifest.csv',index=False); issues_df.to_csv(MANIFEST_DIR/'indiana_manifest_issues.csv',index=False)
frontal=manifest[(manifest.view_guess=='frontal') & manifest.image_exists & manifest.ground_truth_report.ne('')].copy(); frontal.to_csv(MANIFEST_DIR/'indiana_frontal_manifest.csv',index=False)
print('candidate:',len(manifest),'frontal valid:',len(frontal),'lateral:',sum(manifest.view_guess=='lateral'),'unknown:',sum(manifest.view_guess=='unknown'),'issues:',len(issues_df),'paper reference: 3689 pairs')

In [ ]:
# Deterministic, persistent subsets. Existing files are reused.
import numpy as np
for n in [10,50,100]:
    out=SUBSET_DIR/f'indiana_subset_{n}_seed42.csv'
    if not out.exists():
        if len(frontal)<n: print('Not enough valid frontal rows for',n); continue
        frontal.sample(n=n,random_state=SEED).sort_values(['report_id','image_id']).to_csv(out,index=False)
    print(out, len(pd.read_csv(out)))
subset_path=SUBSET_DIR/f'indiana_subset_{RUN_SUBSET_SIZE}_seed42.csv'
assert subset_path.exists(), 'Create/validate the requested subset first'
subset=pd.read_csv(subset_path)

## Official CheXpert Labeler (isolated environment)
The following cells never install the legacy dependency set into the CXR-LLaVA runtime. A setup or official-sample failure is logged and sets `CHEXPERT_READY=False`; clinical metrics must then remain stopped.

In [ ]:
TOOLS_ROOT=Path('/content/cxr_exp02_tools'); CHEXPERT_LABELER_DIR=TOOLS_ROOT/'chexpert-labeler'; NEGBIO_DIR=TOOLS_ROOT/'NegBio'; TOOLS_ROOT.mkdir(exist_ok=True)
for url,d in [('https://github.com/stanfordmlgroup/chexpert-labeler.git',CHEXPERT_LABELER_DIR),('https://github.com/ncbi-nlp/NegBio.git',NEGBIO_DIR)]:
    if not d.exists(): subprocess.run(['git','clone',url,str(d)],check=True)
assert (CHEXPERT_LABELER_DIR/'label.py').exists() and (CHEXPERT_LABELER_DIR/'environment.yml').exists() and NEGBIO_DIR.exists()
(LOG_DIR/'chexpert_labeler_version.txt').write_text(subprocess.check_output(['git','-C',str(CHEXPERT_LABELER_DIR),'rev-parse','HEAD'],text=True)+'\n'+subprocess.check_output(['git','-C',str(NEGBIO_DIR),'rev-parse','HEAD'],text=True),encoding='utf-8')
print(CHEXPERT_LABELER_DIR, NEGBIO_DIR)

In [ ]:
# Run exactly the roadmap's isolated micromamba setup and official sample gate.
MICRO=Path('/content/micromamba-bin/bin/micromamba'); MICRO_ROOT='/content/micromamba-root'; CHEXPERT_READY=False
try:
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','default-jre'],check=True)
    Path('/content/micromamba-bin').mkdir(exist_ok=True)
    if not MICRO.exists(): subprocess.run('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/micromamba-bin bin/micromamba',shell=True,check=True)
    subprocess.run([str(MICRO),'create','-y','-r',MICRO_ROOT,'-n','chexpert-label','-f',str(CHEXPERT_LABELER_DIR/'environment.yml')],check=True)
    base=[str(MICRO),'run','-r',MICRO_ROOT,'-n','chexpert-label']
    envp=os.environ.copy(); envp['PYTHONPATH']=str(NEGBIO_DIR)
    subprocess.run(base+['python','-m','nltk.downloader','universal_tagset','punkt','wordnet'],env=envp,check=True)
    subprocess.run(base+['python','-c',"from bllipparser import RerankingParser; RerankingParser.fetch_and_load('GENIA+PubMed')"],env=envp,check=True)
    subprocess.run(base+['python',str(CHEXPERT_LABELER_DIR/'label.py'),'--reports_path',str(CHEXPERT_LABELER_DIR/'sample_reports.csv'),'--output_path','/content/chexpert_sample_labeled.csv','--verbose'],env=envp,check=True)
    assert Path('/content/chexpert_sample_labeled.csv').exists()
    CHEXPERT_READY=True
except Exception as e:
    (LOG_DIR/'chexpert_setup_error.txt').write_text(repr(e),encoding='utf-8')
    print('CheXpert setup/sample gate failed; clinical metrics are stopped:',repr(e))
print('CHEXPERT_READY:',CHEXPERT_READY)

In [ ]:
# Reuse the baseline's proven NF4/T4 loading and chat-template compatibility patch.
if RUN_INFERENCE:
    import transformers
    from transformers import AutoModel, BitsAndBytesConfig
    from unittest.mock import patch
    if not torch.cuda.is_available(): raise RuntimeError("Enable a Colab GPU before inference.")
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16,llm_int8_skip_modules=["vision_tower","mm_projector","lm_head"])
    original=transformers.LlamaTokenizer.from_pretrained
    def patched(*args,**kwargs): kwargs.pop("add_special_tokens",None); return original(*args,**kwargs)
    with patch.object(transformers.LlamaTokenizer,"from_pretrained",new=patched):
        model=AutoModel.from_pretrained(MODEL_ID,trust_remote_code=True,torch_dtype=torch.float16,quantization_config=q,low_cpu_mem_usage=True,device_map={"":0})
    model.eval()
    if not model.tokenizer.chat_template: model.tokenizer.chat_template = "{% for message in messages %}{{ message['content'] }}{% endfor %}"
else: print("Inference disabled. Set RUN_INFERENCE=True explicitly after reviewing paths and subset.")

In [ ]:
# Incremental/resumable report inference.
prediction_path=PREDICTION_DIR/f'indiana_subset_{RUN_SUBSET_SIZE}_predictions.csv'
if RUN_INFERENCE:
    previous=pd.read_csv(prediction_path) if prediction_path.exists() else pd.DataFrame()
    done=set(previous.loc[previous.status=='success','image_id']) if len(previous) else set()
    records=previous.to_dict('records') if len(previous) else []
    for row in subset.to_dict('records'):
        if row['image_id'] in done: continue
        rec={k:row[k] for k in ['report_id','image_id','image_path','view_guess','ground_truth_report']}
        try:
            image=Image.open(row['image_path']).convert('L'); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); start=time.perf_counter()
            with torch.inference_mode(): generated=model.write_radiologic_report(image)
            torch.cuda.synchronize(); rec.update(generated_report=str(generated),latency_sec=time.perf_counter()-start,peak_vram_gb=torch.cuda.max_memory_allocated()/1024**3,status='success',error_message='')
        except Exception as e: rec.update(generated_report='',latency_sec=None,peak_vram_gb=None,status='failed',error_message=repr(e))
        records.append(rec); pd.DataFrame(records).to_csv(prediction_path,index=False); print(row['image_id'],rec['status'],flush=True)
    pd.DataFrame(records).to_csv(prediction_path,index=False)
else: print('No inference executed; no prediction artifact was created.')

## CheXpert labeling and metrics
Run these cells only after `RUN_INFERENCE=True`, a successful 10-image inference check, and then the intended 50-image gate. The cell below deliberately refuses to run clinical metrics unless the official sample gate passed.

In [ ]:
if not RUN_INFERENCE or not CHEXPERT_READY: raise RuntimeError('Clinical metrics stopped: enable inference and pass official CheXpert sample gate.')
pred=pd.read_csv(prediction_path); successful=pred[pred.status=='success'].reset_index(drop=True); assert successful.generated_report.ne('').all()
prefix=f'indiana_subset_{RUN_SUBSET_SIZE}'; gt_in=CHEXPERT_INPUT_DIR/f'{prefix}_gt_reports.csv'; pr_in=CHEXPERT_INPUT_DIR/f'{prefix}_generated_reports.csv'; rowmap=CHEXPERT_INPUT_DIR/f'{prefix}_row_map.csv'
successful[['ground_truth_report']].to_csv(gt_in,index=False,header=False,quoting=csv.QUOTE_ALL); successful[['generated_report']].to_csv(pr_in,index=False,header=False,quoting=csv.QUOTE_ALL)
successful.assign(chexpert_row=range(len(successful)))[['chexpert_row','report_id','image_id']].to_csv(rowmap,index=False)
base=[str(MICRO),'run','-r',MICRO_ROOT,'-n','chexpert-label']; envp=os.environ.copy(); envp['PYTHONPATH']=str(NEGBIO_DIR)
for inp,out in [(gt_in,CHEXPERT_LABEL_DIR/f'{prefix}_gt_chexpert_labels.csv'),(pr_in,CHEXPERT_LABEL_DIR/f'{prefix}_generated_chexpert_labels.csv')]: subprocess.run(base+['python',str(CHEXPERT_LABELER_DIR/'label.py'),'--reports_path',str(inp),'--output_path',str(out),'--verbose'],env=envp,check=True); assert len(pd.read_csv(out))==len(successful)
print('CheXpert inputs and labels saved for',len(successful),'successful reports')

In [ ]:
TARGET_PATHOLOGIES=['Cardiomegaly','Consolidation','Edema','Lung Opacity','Pleural Effusion','Pneumonia','Pneumothorax']
# Metrics are intentionally explicit: only joint definite 0/1 labels are included.
prefix=f'indiana_subset_{RUN_SUBSET_SIZE}'; gt=pd.read_csv(CHEXPERT_LABEL_DIR/f'{prefix}_gt_chexpert_labels.csv'); pr=pd.read_csv(CHEXPERT_LABEL_DIR/f'{prefix}_generated_chexpert_labels.csv'); out=[]
for p in TARGET_PATHOLOGIES:
    valid=gt[p].isin([0,1]) & pr[p].isin([0,1]); g=gt.loc[valid,p]; y=pr.loc[valid,p]; tp=int(((g==1)&(y==1)).sum()); fp=int(((g==0)&(y==1)).sum()); tn=int(((g==0)&(y==0)).sum()); fn=int(((g==1)&(y==0)).sum()); precision=tp/(tp+fp) if tp+fp else float('nan'); recall=tp/(tp+fn) if tp+fn else float('nan'); f1=2*precision*recall/(precision+recall) if precision+recall else float('nan'); out.append({'pathology':p,'n_valid':int(valid.sum()),'n_gt_positive':int((g==1).sum()),'n_gt_negative':int((g==0).sum()),'tp':tp,'fp':fp,'tn':tn,'fn':fn,'precision':precision,'recall':recall,'f1':f1})
metrics=pd.DataFrame(out); metrics.to_csv(METRIC_DIR/f'{prefix}_pathology_metrics.csv',index=False); print(metrics); print('mean_pathology_f1:',metrics.f1.mean())

## Gates and reporting
Before 50/100/full runs, manually verify image/report pairing, non-empty saved outputs, latency/VRAM, CheXpert row alignment, and representative FP/FN cases. Classify 10 images as smoke evaluation and 50/100 as subset external evaluation. Do not call this a full benchmark unless the paper's 3,689-pair relation is reconstructed and documented.